<a href="https://colab.research.google.com/github/SiracencoSerghei/Bootcamp_Python/blob/master/day_73/Programming_Languages.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
# **Goal:** Analyze the popularity of different programming languages ​​over time
---

StackOverflow will help us answer this burning question. Every post on Stack OverFlow has a tag. And that tag can be the name of a programming language.

To determine which language is the most popular, all we need to do is count the number of posts on Stack Overflow that are tagged with each language. The language with the most posts wins!

Today you will learn:

How to visualize data and create charts with Matplotlib

How to summarize, group, and manipulate your data with Pandas to get it in the format you want

How to work with timestamps and time series data

How to create and customize a line chart to your liking

## Get the Data

Either use the provided .csv file or (optionally) get fresh (the freshest?) data from running an SQL query on StackExchange:

Follow this link to run the query from [StackExchange](https://data.stackexchange.com/stackoverflow/query/675441/popular-programming-languages-per-over-time-eversql-com) to get your own .csv file

<code>
select dateadd(month, datediff(month, 0, q.CreationDate), 0) m, TagName, count(*)
from PostTags pt
join Posts q on q.Id=pt.PostId
join Tags t on t.Id=pt.TagId
where TagName in ('java','c','c++','python','c#','javascript','assembly','php','perl','ruby','visual basic','swift','r','object-c','scratch','go','swift','delphi')
and q.CreationDate < dateadd(month, datediff(month, 0, getdate()), 0)
group by dateadd(month, datediff(month, 0, q.CreationDate), 0), TagName
order by dateadd(month, datediff(month, 0, q.CreationDate), 0)
</code>

---
# **Import Statements**
---

In [20]:
import pandas as pd


In [21]:
from google.colab import files
uploaded = files.upload()

Saving QueryResults.csv to QueryResults (1).csv


---
# Data Exploration
---

**Challenge**: Read the .csv file and store it in a Pandas dataframe

In [22]:
df = pd.read_csv('QueryResults.csv')

**Challenge**: Examine the first 5 rows and the last 5 rows of the of the dataframe

In [23]:
df.head()

,m,TagName,Unnamed: 2
0,2008-07-01 00:00:00,c#,3
1,2008-08-01 00:00:00,assembly,7
2,2008-08-01 00:00:00,c,78
3,2008-08-01 00:00:00,c#,492
4,2008-08-01 00:00:00,c++,157


In [24]:
df.tail()

,m,TagName,Unnamed: 2
2727,2024-12-01 00:00:00,php,498
2728,2024-12-01 00:00:00,python,2984
2729,2024-12-01 00:00:00,r,575
2730,2024-12-01 00:00:00,ruby,54
2731,2024-12-01 00:00:00,swift,357


**Challenge:** try to provide these column names: ['DATE', 'TAG', 'POSTS']

In [31]:
# df.rename(columns={'m': 'DATE', 'TagName': 'TAG', 'Unnamed: 2': 'POSTS'}, inplace=True)
# df = df.rename(columns={'m': 'DATE', 'TagName': 'TAG', 'Unnamed: 2': 'POSTS'})
df.columns = ['DATE', 'TAG', 'POSTS']
df.head()

,DATE,TAG,POSTS
0,2008-07-01 00:00:00,c#,3
1,2008-08-01 00:00:00,assembly,7
2,2008-08-01 00:00:00,c,78
3,2008-08-01 00:00:00,c#,492
4,2008-08-01 00:00:00,c++,157


**Challenge:** Check how many rows and how many columns there are.
What are the dimensions of the dataframe?

In [29]:
print( f'The dimensions of the dataframe is {df.shape[0]} rows and {df.shape[1]} columns')

The dimensions of the dataframe is 2732 rows and 3 columns


**Challenge**: Count the number of entries in each column of the dataframe

To count the number of entries in each column we can use .count().

Note that .count() will actually tell us the number of non-NaN values in each column.

In [30]:
column_counts = df.count()
column_counts

,0
DATE,2732
TAG,2732
POSTS,2732


**Challenge**: Calculate the total number of post per language.
Which Programming language has had the highest total number of posts of all time?

In order to look at the number of entries and the number of posts by programming language, we need to make use of the .groupby() method.

The key is combining .groupby() with the TAG column, which holds as our categories (the names of the programming languages).

If we .sum() the number of posts then we can see how many posts each programming language had since the creation of Stack Overflow.

In [38]:
grouped_df = df.groupby('TAG').sum()
grouped_df = grouped_df.drop(columns='DATE')
grouped_df



,POSTS
TAG,
assembly,43425
c,393714
c#,1584567
c++,790733
delphi,50896
go,72723
java,1873598
javascript,2462326
perl,66983


Some languages are older (e.g., C) and other languages are newer (e.g., Swift). The dataset starts in September 2008.

**Challenge**: How many months of data exist per language? Which language had the fewest months with an entry?


In [39]:
grouped_df = df.groupby('TAG').count()
grouped_df

,DATE,POSTS
TAG,,
assembly,197,197
c,197,197
c#,198,198
c++,197,197
delphi,197,197
go,182,182
java,197,197
javascript,197,197
perl,197,197


---
# Data Cleaning
---

Let's fix the date format to make it more readable. We need to use Pandas to change format from a string of "2008-07-01 00:00:00" to a datetime object with the format of "2008-07-01"

---
# Data Manipulation
---

**Challenge**: What are the dimensions of our new dataframe? How many rows and columns does it have? Print out the column names and print out the first 5 rows of the dataframe.

**Challenge**: Count the number of entries per programming language. Why might the number of entries be different?

---
# Data Visualisaton with with Matplotlib
---

**Challenge**: Use the [matplotlib documentation](https://matplotlib.org/3.2.1/api/_as_gen/matplotlib.pyplot.plot.html#matplotlib.pyplot.plot) to plot a single programming language (e.g., java) on a chart.

**Challenge**: Show two line (e.g. for Java and Python) on the same chart.

---
# Smoothing out Time Series Data
---

Time series data can be quite noisy, with a lot of up and down spikes. To better see a trend we can plot an average of, say 6 or 12 observations. This is called the rolling mean. We calculate the average in a window of time and move it forward by one overservation. Pandas has two handy methods already built in to work this out: [rolling()](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rolling.html) and [mean()](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.core.window.rolling.Rolling.mean.html).